In [1]:
# Importar las librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
# URL del archivo en el bucket público de AWS
url = "https://hybridge-education-machine-learning-datasets.s3.us-east-1.amazonaws.com/Fraud.csv"
df = pd.read_csv(url)

In [3]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [4]:
len(df)

6362620

In [5]:
# Buscar elementos null 
print("valores faltantes por columna")
df.isnull().sum()

valores faltantes por columna


step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

**No hay valores null, omitimos proceso de sustitución de valores null**

In [6]:
# ¿Qué columnas son de tipo object? 
df.select_dtypes(include=['object']).columns

Index(['type', 'nameOrig', 'nameDest'], dtype='object')

In [7]:
# ¿Qué valores puede tomar "type"
df['type'].value_counts()

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

In [8]:
# Eliminar las columnas de identificadores que no aportan valor predictivo directo
df_clean = df.drop(columns=['nameOrig', 'nameDest'])
df_clean.sample(5)

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
5439096,378,CASH_OUT,263218.63,1059.0,0.00,41137.93,304356.56,0,0
1783430,162,CASH_OUT,373690.98,57783.0,0.00,249791.31,623482.29,0,0
4242019,306,CASH_OUT,146925.22,0.0,0.00,389718.48,536643.70,0,0
4353269,309,PAYMENT,6029.76,19930.0,13900.24,0.00,0.00,0,0
1887015,164,CASH_OUT,73711.46,47409.0,0.00,0.00,73711.46,0,0


In [9]:
# Codificar la variable 'type' en variables dummy (0 y 1)
df_encoded = pd.get_dummies(df_clean, columns=['type'], drop_first=True, dtype=int)
df_encoded.sample(5)

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
5032171,354,121220.59,0.00,0.00,286073.58,407294.17,0,0,1,0,0,0
3809162,281,56693.92,0.00,0.00,372884.79,429578.72,0,0,0,0,0,1
833252,41,13970.16,0.00,0.00,0.00,0.00,0,0,0,0,1,0
1755433,161,171938.70,3206580.44,3378519.15,2015595.66,1843656.96,0,0,0,0,0,0
3186055,239,11526.03,96592.47,85066.44,0.00,0.00,0,0,0,0,1,0


In [10]:
# Separar características (X) y variable objetivo (y)
X = df_encoded.drop(columns=['isFraud'])
Y = df_encoded['isFraud']

# Dividir sets de entrenamiento y prueba 
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=42)
print(f"Tamaño de entrenamiento: {x_train.shape}")
print(f"Tamaño de prueba: {x_test.shape}")

Tamaño de entrenamiento: (5090096, 11)
Tamaño de prueba: (1272524, 11)


In [12]:
#Inicializar scaler
scaler = StandardScaler()
# Ajustar y transformar el set de entrenamiento; solo transformar el de prueba
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [13]:
# Inicializar modelo
model = LogisticRegression(random_state=42, max_iter=1000)

In [14]:
# Entrerar el modelo
model.fit(x_train_scaled, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [15]:
# obtener predicciones
y_pred = model.predict(x_test_scaled)


In [16]:
# Matriz de confusión
print("Matriz de Confusión:")
print(confusion_matrix(y_test, y_pred))

# Métricas detalladas por clase
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred))

Matriz de Confusión:
[[1270839      42]
 [    990     653]]

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.94      0.40      0.56      1643

    accuracy                           1.00   1272524
   macro avg       0.97      0.70      0.78   1272524
weighted avg       1.00      1.00      1.00   1272524

